## 1. Импорты и настройки

Нам понадобятся только `numpy` и `pandas`.  
`scipy.stats` используем один раз в конце, чтобы аккуратно посчитать доверительный интервал.

In [18]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 2. Генерируем A/B-тест со стратами

Будем считать, что у пользователей есть pre-treatment сегмент:

- `new` — новые пользователи;
- `regular` — обычные пользователи;
- `vip` — крупные пользователи.

Сегмент известен **до эксперимента**, поэтому его можно использовать как страту.

In [19]:
rng = np.random.default_rng(42)

n = 20_000

segment = rng.choice(
    ['new', 'regular', 'vip'],
    size=n,
    p=[0.50, 0.35, 0.15]
)

variant = rng.integers(0, 2, size=n)

baseline = np.select(
    [segment == 'new', segment == 'regular', segment == 'vip'],
    [4.0, 8.0, 18.0]
)

true_effect = 1.0
noise = rng.normal(loc=0.0, scale=5.0, size=n)

y = baseline + true_effect * variant + noise

exp = pd.DataFrame({
    'segment': segment,
    'variant': variant,
    'y': y,
})

exp.head()

,segment,variant,y
0,regular,1,-0.4900
1,new,1,0.7707
2,vip,0,13.0288
3,regular,1,5.6485
4,new,1,6.4994


Проверим размеры страт и баланс treatment/control внутри каждой страты.

In [20]:
summary = (
    exp
    .groupby(['segment', 'variant'])
    .size()
    .unstack()
    .rename(columns={0: 'control', 1: 'treatment'})
)

summary['total'] = summary['control'] + summary['treatment']
summary['treatment_share'] = summary['treatment'] / summary['total']
summary

variant,control,treatment,total,treatment_share
segment,,,,
new,4945,4993,9938,0.5024
regular,3567,3523,7090,0.4969
vip,1487,1485,2972,0.4997


## 3. Baseline: обычная оценка эффекта

Наивная оценка эффекта — это обычная разница средних:

$$
\widehat{\tau}_{\mathrm{naive}}
=
\bar{Y}_{\mathrm{treatment}}
-
\bar{Y}_{\mathrm{control}}
$$

Стандартная ошибка:

$$
\operatorname{SE}(\widehat{\tau}_{\mathrm{naive}})
=
\sqrt{
\frac{s_{\mathrm{treatment}}^2}{n_{\mathrm{treatment}}}
+
\frac{s_{\mathrm{control}}^2}{n_{\mathrm{control}}}
}
$$

In [28]:
def diff_in_means(metric: pd.Series, variant: pd.Series) -> dict:
    treatment = metric[variant == 1]
    control = metric[variant == 0]

    effect = treatment.mean() - control.mean()

    variance = treatment.var(ddof=1) / len(treatment) + control.var(ddof=1) / len(control)
    se = np.sqrt(variance)

    return {
        'effect': effect,
        'se': se,
        'ci_left': effect - 1.96 * se,
        'ci_right': effect + 1.96 * se,
    }

naive_result = diff_in_means(exp['y'], exp['variant'])
naive_result

{'effect': np.float64(0.8424215537083706),
 'se': np.float64(0.09747414312146647),
 'ci_left': np.float64(0.6513722331902962),
 'ci_right': np.float64(1.033470874226445)}

## 4. Идея стратифицированной оценки

Сначала считаем эффект внутри каждой страты:

$$
\widehat{\tau}_{h}
=
\bar{Y}_{\mathrm{treatment}, h}
-
\bar{Y}_{\mathrm{control}, h}
$$

Потом собираем общий эффект как взвешенную сумму:

$$
\widehat{\tau}_{\mathrm{strat}}
=
\sum_{h=1}^{H} w_h \widehat{\tau}_{h}
$$

где вес страты равен ее доле в популяции:

$$
w_h
=
\frac{N_h}{N},
\qquad
\sum_{h=1}^{H} w_h = 1
$$

## 5. Считаем стратифицированный estimator руками


In [22]:
rows = []

for h, part in exp.groupby('segment'):
    treatment = part.loc[part['variant'] == 1, 'y']
    control = part.loc[part['variant'] == 0, 'y']

    weight_h = len(part) / len(exp)

    tau_h = treatment.mean() - control.mean()

    var_h = treatment.var(ddof=1) / len(treatment) + control.var(ddof=1) / len(control)

    rows.append({
        'segment': h,
        'n': len(part),
        'weight': weight_h,
        'tau_h': tau_h,
        'var_h': var_h,
        'se_h': np.sqrt(var_h),
    })

strata_table = pd.DataFrame(rows)
strata_table

,segment,n,weight,tau_h,var_h,se_h
0,new,9938,0.4969,0.8454,0.0102,0.1009
1,regular,7090,0.3545,0.9647,0.0141,0.1189
2,vip,2972,0.1486,0.6797,0.0328,0.1811


In [23]:
effect_strat = np.sum(strata_table['weight'] * strata_table['tau_h'])

var_strat = np.sum((strata_table['weight'] ** 2) * strata_table['var_h'])
se_strat = np.sqrt(var_strat)

strat_result = {
    'effect': effect_strat,
    'se': se_strat,
    'ci_left': effect_strat - 1.96 * se_strat,
    'ci_right': effect_strat + 1.96 * se_strat,
}

strat_result

{'effect': np.float64(0.8630802064454596),
 'se': np.float64(0.07080602464304178),
 'ci_left': np.float64(0.7243003981450977),
 'ci_right': np.float64(1.0018600147458214)}

## 6. Сравним naive и stratified

Теперь сравним эффект, стандартную ошибку и ширину доверительного интервала.

In [24]:
comparison = pd.DataFrame([
    {'method': 'naive', **naive_result},
    {'method': 'stratified', **strat_result},
])

comparison['ci_width'] = comparison['ci_right'] - comparison['ci_left']
comparison['se_reduction_vs_naive'] = 1 - comparison['se'] / comparison.loc[comparison['method'] == 'naive', 'se'].iloc[0]
comparison

,method,effect,se,ci_left,ci_right,ci_width,se_reduction_vs_naive
0,naive,0.8424,0.0975,0.6514,1.0335,0.3821,0.0000
1,stratified,0.8631,0.0708,0.7243,1.0019,0.2776,0.2736


Если страты хорошо объясняют уровень метрики, стратифицированная оценка часто имеет меньшую стандартную ошибку.

Важно: это не способ «улучшить p-value». Это способ сделать оценку эффекта более точной за счет сравнения treatment/control внутри похожих групп.

## 7. Проверяем p-value и доверительный интервал

Для простоты используем нормальное приближение:

$$
z
=
\frac{\widehat{\tau}}{\operatorname{SE}(\widehat{\tau})}
$$

In [25]:
def add_p_value(row):
    z = row['effect'] / row['se']
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return pd.Series({'z': z, 'p_value': p_value})

comparison[['z', 'p_value']] = comparison.apply(add_p_value, axis=1)
comparison

,method,effect,se,ci_left,ci_right,ci_width,se_reduction_vs_naive,z,p_value
0,naive,0.8424,0.0975,0.6514,1.0335,0.3821,0.0000,8.6425,0.0000
1,stratified,0.8631,0.0708,0.7243,1.0019,0.2776,0.2736,12.1894,0.0000


## 8. Что будет, если страта плохая

Создадим случайную страту, которая никак не связана с метрикой.  
Обычно такая стратификация почти не уменьшает SE.

In [26]:
exp_bad = exp.copy()
exp_bad['random_stratum'] = rng.choice(['A', 'B', 'C'], size=len(exp_bad))

def stratified_estimator(data: pd.DataFrame, metric_col: str, variant_col: str, stratum_col: str) -> dict:
    rows = []
    for h, part in data.groupby(stratum_col):
        treatment = part.loc[part[variant_col] == 1, metric_col]
        control = part.loc[part[variant_col] == 0, metric_col]
        weight_h = len(part) / len(data)
        tau_h = treatment.mean() - control.mean()
        var_h = treatment.var(ddof=1) / len(treatment) + control.var(ddof=1) / len(control)
        rows.append((weight_h, tau_h, var_h))

    tmp = pd.DataFrame(rows, columns=['weight', 'tau_h', 'var_h'])
    effect = np.sum(tmp['weight'] * tmp['tau_h'])
    se = np.sqrt(np.sum(tmp['weight'] ** 2 * tmp['var_h']))
    return {'effect': effect, 'se': se}

bad_result = stratified_estimator(exp_bad, 'y', 'variant', 'random_stratum')
bad_result

{'effect': np.float64(0.8425098106018413),
 'se': np.float64(0.09748942536907197)}

In [27]:
pd.DataFrame([
    {'method': 'naive', 'effect': naive_result['effect'], 'se': naive_result['se']},
    {'method': 'good_strata_segment', 'effect': strat_result['effect'], 'se': strat_result['se']},
    {'method': 'bad_random_strata', 'effect': bad_result['effect'], 'se': bad_result['se']},
])

,method,effect,se
0,naive,0.8424,0.0975
1,good_strata_segment,0.8631,0.0708
2,bad_random_strata,0.8425,0.0975


# Дополнение: почему стратификация снижает дисперсию

До этого мы говорили, что стратификация может сделать оценку эффекта более точной.

Но почему это вообще происходит?

Идея довольно простая: общая вариативность метрики состоит из двух частей:

1. различий пользователей **внутри одной страты**;
2. различий **между самими стратами**.

При обычной рандомизации treatment и control могут случайно получить немного разный состав страт.

Например, в treatment случайно может оказаться больше пользователей с высоким базовым уровнем метрики, а в control — больше пользователей с низким уровнем метрики.

Такой дисбаланс создаёт дополнительный шум.

Стратифицированная рандомизация устраняет этот источник случайного шума, потому что treatment и control балансируются отдельно внутри каждой страты.

Ниже покажем это формально.

---

## 1. Обозначения

Пусть $H$ — номер страты пользователя.

Например,

$$
H \in \{1,2,\ldots,K\}.
$$

Обозначим долю пользователей, принадлежащих страте $h$, как

$$
w_h
=
\frac{n_h}{n},
$$

где:

- $n_h$ — количество пользователей в страте $h$;
- $n$ — общее количество пользователей.

При этом

$$
\sum_{h=1}^{K} w_h = 1.
$$

Пусть $Y$ — наша целевая метрика.

Для простоты будем считать, что доля treatment во всех стратах одинакова и равна $p$.

Тогда

$$
n_T = pn,
$$

а

$$
n_C = (1-p)n.
$$

Также для простоты предполагаем стандартную ситуацию, в которой treatment не меняет дисперсию метрики радикально.

Нас будет интересовать именно дополнительная дисперсия, возникающая из-за неоднородности пользователей между стратами.

---

## 2. Закон полной дисперсии

Для случайной величины $Y$ и страты $H$ выполняется закон полной дисперсии:

$$
\operatorname{Var}(Y)
=
\mathbb{E}
\left[
\operatorname{Var}(Y \mid H)
\right]
+
\operatorname{Var}
\left(
\mathbb{E}[Y \mid H]
\right).
$$

Это ключевая формула.

Она говорит, что общая дисперсия метрики состоит из двух компонентов.

Первый компонент:

$$
\mathbb{E}
\left[
\operatorname{Var}(Y \mid H)
\right]
$$

— это средняя дисперсия **внутри страт**.

То есть насколько различаются пользователи, если мы уже знаем, к какой страте они относятся.

Второй компонент:

$$
\operatorname{Var}
\left(
\mathbb{E}[Y \mid H]
\right)
$$

— это дисперсия **между стратами**.

То есть насколько отличаются средние значения метрики в разных стратах.

Поэтому можно запомнить:

$$
\boxed{
\text{общая дисперсия}
=
\text{дисперсия внутри страт}
+
\text{дисперсия между стратами}
}
$$

---

## 3. Что происходит при обычной рандомизации

Без стратификации мы оцениваем эффект стандартной разницей средних:

$$
\widehat{\tau}_{\mathrm{ordinary}}
=
\bar{Y}_{T}
-
\bar{Y}_{C}.
$$

При размерах групп

$$
n_T = pn
$$

и

$$
n_C = (1-p)n
$$

дисперсия разницы средних имеет вид

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{ordinary}}
\right)
=
\frac{\operatorname{Var}(Y)}{n_T}
+
\frac{\operatorname{Var}(Y)}{n_C}.
$$

Подставим размеры групп:

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{ordinary}}
\right)
=
\frac{\operatorname{Var}(Y)}{pn}
+
\frac{\operatorname{Var}(Y)}{(1-p)n}.
$$

Объединяем:

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{ordinary}}
\right)
=
\frac{
\operatorname{Var}(Y)
}{
np(1-p)
}.
$$

Теперь воспользуемся законом полной дисперсии:

$$
\operatorname{Var}(Y)
=
\mathbb{E}
\left[
\operatorname{Var}(Y \mid H)
\right]
+
\operatorname{Var}
\left(
\mathbb{E}[Y \mid H]
\right).
$$

Получаем

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{ordinary}}
\right)
=
\frac{
\mathbb{E}
\left[
\operatorname{Var}(Y \mid H)
\right]
+
\operatorname{Var}
\left(
\mathbb{E}[Y \mid H]
\right)
}{
np(1-p)
}.
$$

То есть при обычной рандомизации в дисперсию оценки попадают:

- различия пользователей внутри страт;
- различия между самими стратами.

---

## 4. Что меняется при стратификации

Теперь предположим, что мы проводим рандомизацию отдельно внутри каждой страты.

В каждой страте $h$ считаем свой эффект:

$$
\widehat{\tau}_h
=
\bar{Y}_{T,h}
-
\bar{Y}_{C,h}.
$$

После этого объединяем эффекты с помощью весов:

$$
\widehat{\tau}_{\mathrm{strat}}
=
\sum_{h=1}^{K}
w_h
\widehat{\tau}_h.
$$

Вес каждой страты равен её доле в общей выборке:

$$
w_h
=
\frac{n_h}{n}.
$$

---

## 5. Дисперсия оценки внутри одной страты

Обозначим дисперсию метрики внутри страты $h$ как

$$
\sigma_h^2
=
\operatorname{Var}(Y \mid H=h).
$$

Если доля treatment внутри каждой страты равна $p$, то

$$
n_{T,h}
=
pn_h
$$

и

$$
n_{C,h}
=
(1-p)n_h.
$$

Тогда дисперсия разницы средних внутри страты равна

$$
\operatorname{Var}
\left(
\widehat{\tau}_h
\right)
=
\frac{\sigma_h^2}{pn_h}
+
\frac{\sigma_h^2}{(1-p)n_h}.
$$

Объединяем выражение:

$$
\operatorname{Var}
\left(
\widehat{\tau}_h
\right)
=
\frac{
\sigma_h^2
}{
n_h p(1-p)
}.
$$

---

## 6. Дисперсия итоговой стратифицированной оценки

Итоговая оценка имеет вид

$$
\widehat{\tau}_{\mathrm{strat}}
=
\sum_{h=1}^{K}
w_h
\widehat{\tau}_h.
$$

Поскольку рандомизация происходит независимо внутри разных страт, дисперсия взвешенной суммы равна

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{strat}}
\right)
=
\sum_{h=1}^{K}
w_h^2
\operatorname{Var}
\left(
\widehat{\tau}_h
\right).
$$

Подставляем выражение для дисперсии внутри страты:

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{strat}}
\right)
=
\sum_{h=1}^{K}
w_h^2
\frac{
\sigma_h^2
}{
n_h p(1-p)
}.
$$

Но

$$
n_h = nw_h.
$$

Поэтому

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{strat}}
\right)
=
\sum_{h=1}^{K}
w_h^2
\frac{
\sigma_h^2
}{
nw_h p(1-p)
}.
$$

Один $w_h$ сокращается:

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{strat}}
\right)
=
\frac{
\sum_{h=1}^{K}
w_h\sigma_h^2
}{
np(1-p)
}.
$$

Но

$$
\sum_{h=1}^{K}
w_h\sigma_h^2
=
\mathbb{E}
\left[
\operatorname{Var}(Y \mid H)
\right].
$$

Следовательно,

$$
\boxed{
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{strat}}
\right)
=
\frac{
\mathbb{E}
\left[
\operatorname{Var}(Y \mid H)
\right]
}{
np(1-p)
}
}
$$

---

## 7. Сравним две дисперсии

Для обычной оценки мы получили:

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{ordinary}}
\right)
=
\frac{
\mathbb{E}
\left[
\operatorname{Var}(Y \mid H)
\right]
+
\operatorname{Var}
\left(
\mathbb{E}[Y \mid H]
\right)
}{
np(1-p)
}.
$$

Для стратифицированной оценки:

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{strat}}
\right)
=
\frac{
\mathbb{E}
\left[
\operatorname{Var}(Y \mid H)
\right]
}{
np(1-p)
}.
$$

Вычтем одну дисперсию из другой:

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{ordinary}}
\right)
-
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{strat}}
\right)
=
\frac{
\operatorname{Var}
\left(
\mathbb{E}[Y \mid H]
\right)
}{
np(1-p)
}.
$$

Но любая дисперсия неотрицательна:

$$
\operatorname{Var}
\left(
\mathbb{E}[Y \mid H]
\right)
\geq 0.
$$

Следовательно,

$$
\boxed{
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{strat}}
\right)
\leq
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{ordinary}}
\right)
}
$$

То есть при выполнении наших предположений стратифицированная оценка имеет дисперсию не больше, чем обычная оценка.

---

## 8. Откуда именно появляется выигрыш

Особенно интересно посмотреть на разницу:

$$
\boxed{
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{ordinary}}
\right)
-
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{strat}}
\right)
=
\frac{
\operatorname{Var}
\left(
\mathbb{E}[Y \mid H]
\right)
}{
np(1-p)
}
}
$$

Именно величина

$$
\operatorname{Var}
\left(
\mathbb{E}[Y \mid H]
\right)
$$

определяет потенциальную пользу стратификации.

Если средние значения метрики сильно различаются между стратами, то эта величина большая.

Значит, стратификация может заметно снизить дисперсию.

Например, если мы разделили пользователей на:

- low activity;
- medium activity;
- high activity;

и эти группы действительно сильно отличаются по revenue, то стратификация может дать заметный выигрыш.

---

## 9. А если страты плохие?

Предположим, что средняя метрика практически одинакова во всех стратах.

Тогда

$$
\mathbb{E}[Y \mid H=h]
\approx
\mathbb{E}[Y]
$$

для всех $h$.

Следовательно,

$$
\operatorname{Var}
\left(
\mathbb{E}[Y \mid H]
\right)
\approx 0.
$$

Тогда

$$
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{ordinary}}
\right)
\approx
\operatorname{Var}
\left(
\widehat{\tau}_{\mathrm{strat}}
\right).
$$

То есть стратификация практически ничего нам не даёт.

Поэтому разбивать пользователей на произвольные группы бессмысленно.

Хороший признак для стратификации должен быть связан с целевой метрикой.

---

## Главный вывод

Стратификация уменьшает дисперсию не потому, что мы каким-то образом меняем эффект эксперимента.

Мы по-прежнему оцениваем тот же самый эффект treatment.

Стратификация просто убирает дополнительный шум, который возникает из-за случайного дисбаланса состава treatment и control.

Можно запомнить это следующим образом:

$$
\boxed{
\text{обычная рандомизация}
=
\text{шум внутри страт}
+
\text{шум из-за состава страт}
}
$$

а при стратифицированной рандомизации:

$$
\boxed{
\text{стратификация}
\approx
\text{только шум внутри страт}
}
$$

Именно поэтому стратификация особенно полезна тогда, когда страты хорошо объясняют различия в целевой метрике.

> **Важно:** это доказательство использует стандартную упрощённую постановку: одинаковую долю treatment внутри страт и пропорциональные веса страт. При сложных схемах аллокации формулы становятся более общими, но основная идея остаётся той же.